In [4]:
import os
from pathlib import Path
import requests
from dotenv import load_dotenv

# Load env from project locations
for env_path in [
    Path("server/.env"),
    Path(".env"),
    Path("../server/.env"),
]:
    if env_path.exists():
        load_dotenv(env_path, override=False)

HF_API_KEY = "hf_REDACTED"
BASE_URL = "https://router.huggingface.co/v1/chat/completions"

# Models you requested + one reliable fallback
models_to_test = [
    "meta-llama/Llama-3.2-3B-Instruct",
    "microsoft/Phi-3.5-mini-instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
]

prompt = "Reply with exactly: MODEL_OK"

if not HF_API_KEY:
    print("HUGGINGFACE_API_KEY is not set in this notebook environment.")
    print("Add key in server/.env or run:")
    print("%env HUGGINGFACE_API_KEY=your_token_here")
    print("Then rerun Cell 1.")
else:
    headers = {
        "Authorization": f"Bearer {HF_API_KEY}",
        "Content-Type": "application/json",
    }

    for model in models_to_test:
        try:
            payload = {
                "model": model,
                "messages": [
                    {"role": "system", "content": "You are a test assistant."},
                    {"role": "user", "content": prompt},
                ],
                "max_tokens": 20,
                "temperature": 0,
            }

            resp = requests.post(BASE_URL, headers=headers, json=payload, timeout=45)
            if resp.status_code == 200:
                data = resp.json()
                text = data.get("choices", [{}])[0].get("message", {}).get("content", "").strip()
                print(f"✅ {model}: OK | response='{text}'")
            else:
                print(f"❌ {model}: HTTP {resp.status_code}")
                print(resp.text[:500])
        except Exception as e:
            print(f"❌ {model}: Exception -> {type(e).__name__}: {e}")

✅ meta-llama/Llama-3.2-3B-Instruct: OK | response='MODEL_OK'
❌ microsoft/Phi-3.5-mini-instruct: HTTP 400
{"error":{"message":"The requested model 'microsoft/Phi-3.5-mini-instruct' is not supported by any provider you have enabled.","type":"invalid_request_error","param":"model","code":"model_not_supported"}}
❌ mistralai/Mistral-7B-Instruct-v0.3: HTTP 400
{"error":{"message":"The requested model 'mistralai/Mistral-7B-Instruct-v0.3' is not a chat model.","type":"invalid_request_error","param":"model","code":"model_not_supported"}}


In [5]:
# Probe alternative chat-model IDs on Hugging Face Router
alt_models = [
    "meta-llama/Llama-3.1-8B-Instruct",
    "meta-llama/Llama-3.2-1B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.2",
    "google/gemma-2-2b-it",
]

if not HF_API_KEY:
    print("HUGGINGFACE_API_KEY missing")
else:
    headers = {
        "Authorization": f"Bearer {HF_API_KEY}",
        "Content-Type": "application/json",
    }
    for model in alt_models:
        try:
            payload = {
                "model": model,
                "messages": [
                    {"role": "system", "content": "You are a test assistant."},
                    {"role": "user", "content": "Reply with exactly: MODEL_OK"},
                ],
                "max_tokens": 20,
                "temperature": 0,
            }
            resp = requests.post(BASE_URL, headers=headers, json=payload, timeout=45)
            if resp.status_code == 200:
                txt = resp.json().get("choices", [{}])[0].get("message", {}).get("content", "").strip()
                print(f"✅ {model}: OK | {txt}")
            else:
                print(f"❌ {model}: HTTP {resp.status_code} | {resp.text[:180]}")
        except Exception as e:
            print(f"❌ {model}: Exception -> {type(e).__name__}: {e}")

✅ meta-llama/Llama-3.1-8B-Instruct: OK | MODEL_OK
✅ meta-llama/Llama-3.2-1B-Instruct: OK | MODEL_OK
✅ Qwen/Qwen2.5-7B-Instruct: OK | MODEL_OK
✅ Qwen/Qwen2.5-Coder-7B-Instruct: OK | MODEL_OK
✅ mistralai/Mistral-7B-Instruct-v0.2: OK | MODEL_OK
❌ google/gemma-2-2b-it: HTTP 400 | {"error":{"message":"The requested model 'google/gemma-2-2b-it' is not supported by any provider you have enabled.","type":"invalid_request_error","param":"model","code":"model_not


In [5]:
!pip install bytez

In [7]:
from bytez import Bytez

key = "266fe222b11bb1f03b8a7b6e53ee9604"
sdk = Bytez(key)

# These are the models required by your llmService.js
required_models = [
    "meta-llama/Meta-Llama-3.1-8B-Instruct", # MAIN / CREATIVE
    "meta-llama/Llama-3.2-1B-Instruct",      # ROUTING
    "meta-llama/Llama-3.2-3B-Instruct",      # FALLBACK
    "Qwen/Qwen2.5-Coder-7B-Instruct",        # TOOLS
    "Qwen/Qwen2.5-7B-Instruct",              # SUMMARIZATION
    "mistralai/Mistral-7B-Instruct-v0.2"     # RESEARCH
]

print("Testing required models on Bytez API...\n")

for model_id in required_models:
    try:
        model = sdk.model(model_id)
        results = model.run([
          {
            "role": "user",
            "content": "Reply with exactly: OK"
          }
        ])
        
        if hasattr(results, 'error') and results.error:
            print(f"❌ {model_id}: ERROR | {results.error}")
        else:
            # Extracting the text output
            out_text = results.output if hasattr(results, 'output') else str(results)
            print(f"✅ {model_id}: SUCCESS | {out_text}")
    except Exception as e:
        print(f"❌ {model_id}: EXCEPTION | {e}")

Testing required models on Bytez API...

✅ meta-llama/Meta-Llama-3.1-8B-Instruct: SUCCESS | {'role': 'assistant', 'content': 'OK'}
❌ meta-llama/Llama-3.2-1B-Instruct: ERROR | Please upgrade your account to use this model. Free plans can only use models up to size `sm`. Models larger than `xxl` require an enterprise plan. Please visit https://bytez.com/api/pricing to upgrade!
✅ meta-llama/Llama-3.2-3B-Instruct: SUCCESS | {'role': 'assistant', 'content': 'OK', 'tool_calls': []}
✅ Qwen/Qwen2.5-Coder-7B-Instruct: SUCCESS | {'content': 'OK', 'role': 'assistant', 'tool_calls': []}
✅ Qwen/Qwen2.5-7B-Instruct: SUCCESS | {'role': 'assistant', 'content': 'OK', 'tool_calls': []}
✅ mistralai/Mistral-7B-Instruct-v0.2: SUCCESS | {'role': 'assistant', 'content': ' OK.'}


In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434"

# 1. Check if Ollama is running
try:
    tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()
    models = [m["name"] for m in tags.get("models", [])]
    print(f"✅ Ollama is running | Models installed: {models}\n")
except Exception as e:
    print(f"❌ Ollama is NOT running | {e}")
    print("   Start it with: ollama serve")
    models = []

# 2. Test each installed model
if models:
    for model in models:
        try:
            resp = requests.post(
                f"{OLLAMA_URL}/api/chat",
                json={
                    "model": model,
                    "messages": [
                        {"role": "system", "content": "You are a test assistant."},
                        {"role": "user", "content": "Reply with exactly: MODEL_OK"},
                    ],
                    "stream": False,
                    "options": {"num_predict": 20, "temperature": 0},
                },
                timeout=60,
            )
            if resp.status_code == 200:
                data = resp.json()
                text = data.get("message", {}).get("content", "").strip()
                eval_tokens = data.get("eval_count", "?")
                duration_ms = round(data.get("total_duration", 0) / 1e6)
                print(f"✅ {model}: OK | response='{text}' | tokens={eval_tokens} | {duration_ms}ms")
            else:
                print(f"❌ {model}: HTTP {resp.status_code} | {resp.text[:200]}")
        except Exception as e:
            print(f"❌ {model}: Exception -> {type(e).__name__}: {e}")
